# 1 - Ingestion

## 1.1 — List files in Unity Catalog Volume

In [0]:
import os

volume_path = "/Volumes/hr_catalogue_business_case/hr_raw/hr_data_businesscase/"

files = dbutils.fs.ls(volume_path)
for f in files:
    print(f.name)


## 1.2 — Load CSVs with spark.read

In [0]:

base = "/Volumes/hr_catalogue_business_case/hr_raw/hr_data_businesscase/"

df_absences  = spark.read.option("header", True).option("inferSchema", True).option("sep", ";").csv(base + "ABSENCES.csv")
df_contracts = spark.read.option("header", True).option("inferSchema", True).option("sep", ";").csv(base + "CONTRACT_BASIS.csv")
df_salary    = spark.read.option("header", True).option("inferSchema", True).option("sep", ";").csv(base + "SALARY_STATEMENT.csv")
df_workplan  = spark.read.option("header", True).option("inferSchema", True).option("sep", ";").csv(base + "WORK_PLAN.csv")
df_postcodes = spark.read.option("header", True).option("inferSchema", True).option("sep", ";").csv(base + "POSTCODES.csv")
df_abs_types = spark.read.option("header", True).option("inferSchema", True).option("sep", ";").csv(base + "Absence_Type.csv")

print("All DataFrames loaded")

print("All DataFrames loaded")

## 1.3 — Inspect CSV, data types

In [0]:
dfs = {
    "absences":  df_absences,
    "contracts": df_contracts,
    "salary":    df_salary,
    "workplan":  df_workplan,
    "postcodes": df_postcodes,
    "abs_types": df_abs_types
}

for name, df in dfs.items():
    print("\n" + "="*50)
    print("Dataset: " + name + " | Rows: " + str(df.count()) + " | Cols: " + str(len(df.columns)))
    print("="*50)
    for col_name, dtype in df.dtypes:
        print("  {:<40} {}".format(col_name, dtype))

## 1.4. Cast columns to correct types

### 1.4.1 Abscences

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import to_date, col

schema_absences = StructType([
    StructField("Firm ID",           IntegerType(), True),
    StructField("Department ID",     IntegerType(), True),
    StructField("Category ID",       IntegerType(), True),
    StructField("Person ID",         IntegerType(), True),
    StructField("Year",              IntegerType(), True),
    StructField("Quarter",           IntegerType(), True),
    StructField("Month",             IntegerType(), True),
    StructField("Date",              StringType(),  True),
    StructField("Period",            StringType(),  True),
    StructField("Qty_Illness_Days",  IntegerType(), True),
    StructField("Qty_Z0_Days",       IntegerType(), True),
    StructField("Qty_Z1_Days",       IntegerType(), True),
    StructField("Qty_Z2_Days",       IntegerType(), True),
    StructField("Qty_Z3_Days",       IntegerType(), True),
    StructField("Qty_P0_Days",       IntegerType(), True),
    StructField("Qty_P1_Days",       IntegerType(), True),
    StructField("Qty_P2_Days",       IntegerType(), True),
    StructField("Qty_P3_Days",       IntegerType(), True),
    StructField("Qty_A1_Days",       IntegerType(), True),
    StructField("Qty_A2_Days",       IntegerType(), True),
    StructField("Freq_Z0_Days",      IntegerType(), True),
    StructField("Freq_Z1_Days",      IntegerType(), True),
    StructField("Freq_Z2_Days",      IntegerType(), True),
    StructField("Freq_Z3_Daqs",      IntegerType(), True),
    StructField("Freq_P0_Days",      IntegerType(), True),
    StructField("Freq_P1_Days",      IntegerType(), True),
    StructField("Freq_P2_Days",      IntegerType(), True),
    StructField("Freq_P3_Days",      IntegerType(), True),
    StructField("Freq_A1_Days",      IntegerType(), True),
    StructField("Freq_A2_Days",      IntegerType(), True),
    StructField("Qty_Days_Worked",   DoubleType(),  True),
    StructField("Qty_Working_Days",  DoubleType(),  True),
])

df_absences = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .schema(schema_absences) \
    .csv(base + "ABSENCES.csv") \
    .withColumn("Date", to_date(col("Date"), "MM/dd/yyyy"))

print("absences loaded: " + str(df_absences.count()) + " rows")

### 1.4.2. Contracts

In [0]:
# Sub-step 1.4b — Load contracts with explicit StructType
schema_contracts = StructType([
    StructField("Contract ZIP Code",          StringType(),  True),
    StructField("Firm ID",                    IntegerType(), True),
    StructField("Department ID",              IntegerType(), True),
    StructField("Category ID",                IntegerType(), True),
    StructField("Person ID",                  IntegerType(), True),
    StructField("Contract Start Date",        DateType(),    True),
    StructField("Contract End Date",          DateType(),    True),
    StructField("Company Start Date",         DateType(),    True),
    StructField("Birth Date",                 DateType(),    True),
    StructField("Contract Terminatio Reason", StringType(),  True),
    StructField("Gender",                     StringType(),  True),
    StructField("Nationality",                StringType(),  True),
    StructField("Contract Type",              StringType(),  True),
])

df_contracts = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .option("dateFormat", "MM/dd/yyyy") \
    .schema(schema_contracts) \
    .csv(base + "CONTRACT_BASIS.csv")

print("contracts loaded: " + str(df_contracts.count()) + " rows")

### 1.4.3. SALARY

In [0]:
schema_salary = StructType([
    StructField("FDCP",             StringType(), True),
    StructField("Gross Salary",     DoubleType(), True),
    StructField("Net Salary",       DoubleType(), True),
    StructField("Gross Salary 108", DoubleType(), True),
    StructField("Period",           StringType(), True),
])

df_salary = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .schema(schema_salary) \
    .csv(base + "SALARY_STATEMENT.csv")

print("salary loaded: " + str(df_salary.count()) + " rows")

###  1.4.4. WORK PLAN

In [0]:
schema_workplan = StructType([
    StructField("FDCP",                  StringType(), True),
    StructField("Valid From",            DateType(),   True),
    StructField("Valid To",              DateType(),   True),
    StructField("Working Days per Week", DoubleType(), True),
    StructField("NACE Code",             StringType(), True),
    StructField("NACE Description",      StringType(), True),
])

df_workplan = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .option("dateFormat", "MM/dd/yyyy") \
    .schema(schema_workplan) \
    .csv(base + "WORK_PLAN.csv")

print("workplan loaded: " + str(df_workplan.count()) + " rows")

### 1.4.6. ABS TYPE

In [0]:
schema_abs_types = StructType([
    StructField("Type_Absence",    StringType(), True),
    StructField("Type_Absence_FR", StringType(), True),
])

df_abs_types = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .schema(schema_abs_types) \
    .csv(base + "Absence_Type.csv")

print("abs_types loaded: " + str(df_abs_types.count()) + " rows")

### 1.4.6. postcodes

In [0]:
display(df_postcodes)

In [0]:
schema_postcodes = StructType([
    StructField("PostCode",    IntegerType(),  True),
    StructField("Region_Code", IntegerType(), True),
    StructField("Region",      StringType(),  True),
])

df_postcodes = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .schema(schema_postcodes) \
    .csv(base + "POSTCODES.csv")

print("postcodes loaded: " + str(df_postcodes.count()) + " rows")

## 1.5 Data Quality: Duplicates & Missing Values

In [0]:
dfs = {
    "absences":     df_absences,
    "contracts":    df_contracts,
    "salary":       df_salary,
    "workplan":     df_workplan,
    "postcodes":    df_postcodes,
    "abs_types":    df_abs_types
}

for name, df in dfs.items():
    total = df.count()
    distinct = df.distinct().count()
    duplicates = total - distinct
    print(f"{name} has {total} rows, {distinct} distinct, {duplicates} duplicates")


### 1.5.1. Removing duplicates in salary

In [0]:
df_salary_deduped = df_salary.dropDuplicates()

print(f"Records before : {df_salary.count()}")
print(f"Records after  : {df_salary_deduped.count()}")


### 1.5.2. Removing duplicates in work plan

In [0]:
df_workplan_deduped = df_workplan.dropDuplicates()

print("Records before : " + str(df_workplan.count()))
print("Records after  : " + str(df_workplan_deduped.count()))

#### Notice that the number of rows after is equal to the number of distinct rows we observed in 1.5. 
We now jus t verify for the 4 other CSV

In [0]:
df_absences_deduped  = df_absences.dropDuplicates()
df_contracts_deduped = df_contracts.dropDuplicates()
df_postcodes_deduped = df_postcodes.dropDuplicates()
df_abs_types_deduped = df_abs_types.dropDuplicates()

print("absences  — before: " + str(df_absences.count())  + " | after: " + str(df_absences_deduped.count()))
print("contracts — before: " + str(df_contracts.count()) + " | after: " + str(df_contracts_deduped.count()))
print("postcodes — before: " + str(df_postcodes.count()) + " | after: " + str(df_postcodes_deduped.count()))
print("abs_types — before: " + str(df_abs_types.count()) + " | after: " + str(df_abs_types_deduped.count()))

### 1.5.3 Verify NULL and misssing values

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

datasets_clean = {
    "absences":  df_absences_deduped,
    "contracts": df_contracts_deduped,
    "salary":    df_salary_deduped,
    "workplan":  df_workplan_deduped,
    "postcodes": df_postcodes_deduped,
    "abs_types": df_abs_types_deduped
}

for name, df in datasets_clean.items():
    print("\n=== " + name + " ===")
    null_counts = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    
    for col_name, null_count in null_counts.items():
        if null_count > 0:
            print("  " + col_name + " : " + str(null_count) + " nulls")
    
    if all(v == 0 for v in null_counts.values()):
        print("  No nulls found")

## 1.6 Save in Delta Lake structure

In [0]:
# Rename columns with spaces before saving to Delta
from pyspark.sql import functions as F

def rename_cols(df):
    for col_name in df.columns:
        new_name = col_name.replace(" ", "_")
        df = df.withColumnRenamed(col_name, new_name)
    return df

In [0]:
# Rename + Save to Delta
delta_base = "hr_catalogue_business_case.hr_raw"

rename_cols(df_absences_deduped).write.format("delta").mode("overwrite").saveAsTable(delta_base + ".absences")
rename_cols(df_contracts_deduped).write.format("delta").mode("overwrite").saveAsTable(delta_base + ".contracts")
rename_cols(df_salary_deduped).write.format("delta").mode("overwrite").saveAsTable(delta_base + ".salary")
rename_cols(df_workplan_deduped).write.format("delta").mode("overwrite").saveAsTable(delta_base + ".workplan")
rename_cols(df_postcodes_deduped).write.format("delta").mode("overwrite").saveAsTable(delta_base + ".postcodes")
rename_cols(df_abs_types_deduped).write.format("delta").mode("overwrite").saveAsTable(delta_base + ".absence_types")

print("All Delta tables updated!")